
Geo Data Science with Python,
Prof. Susanna Werth, VT Geosciences

# OpenDAP, PyDAP and Statistics


This notebook is accompanied by the lecture L04 presentation slides.

---


Content:
-------
- **A.** Streaming data with PyDAP & plotting geographical data with Cartopy
- **B.** Descriptive / univariate statistics
- **C.** Multi-variate statistics


---
## Part A: Streaming Data with PyDAP


### Important sources for using PyDAP & retrieving NASA Earthdata:

PyDap
- Pydap documentation: https://pydap.github.io/pydap/en/intro.html

OpenDAP Data Access Protocol
- https://www.opendap.org/
- https://opendap.github.io/documentation/QuickStart.html
- https://www.opendap.org/support/user-documentation

NASA Earthdata

- Login: https://urs.earthdata.nasa.gov/home
- Register: https://disc.gsfc.nasa.gov/data-access
- What Do I Need to Know About Earthdata Login?: https://urs.earthdata.nasa.gov/documentation/what_do_i_need_to_know
- OpenDap & Browser-based access form: https://www.earthdata.nasa.gov/engage/open-data-services-software/earthdata-developer-portal/opendap

Tutorials from Earthdata (these rely on earthaccess module installation)
- GESDISC Tutorials https://github.com/nasa/gesdisc-tutorials
- Earthdata, read NetCDF data in Python: https://disc.gsfc.nasa.gov/information/howto?title=How%20to%20read%20and%20plot%20NetCDF%20MERRA-2%20data%20in%20Python
- Download subset of data: https://disc.gsfc.nasa.gov/information/howto?title=How%20to%20download%20a%20spatial%20and%20variable%20subset%20of%20Level%201B%20data%20using%20OPeNDAP
- GESDISC data recipes: https://disc.gsfc.nasa.gov/information/howto

<div class="alert alert-warning">
    
**NOTE**
For better download speed, I advised to run this notebook on Google Colab. Before working with the notebook, run the following commands in two separate code cells, to install the non-standard packages we need today:
```python
pip install pydap
pip install cartopy
```

The output on Colab might be slightly different than in Jupyter notebook, but it works the same.
</div>

### Setup Earthdata access

To stream and download data from NASA's Earthdata or DAAC servers, we need to generate an authentication file that passes login credentials to NASA. To do that, add your Earthdata login information (username and password) in the placeholder spaces below marede by `<>`. Then execute the cell. To test, if it worked, use the cell below. If you are interested, further details of this are described here: https://opendap.github.io/documentation/tutorials/ClientAuthentication.html#netrc


In [ ]:
%%bash
echo "" > ~/.netrc   # overwrites existing file; for appending change to >>
echo "machine urs.earthdata.nasa.gov" > ~/.netrc
echo "	login <USERNAME>" >> ~/.netrc
echo "	password <PASSWORD>" >> ~/.netrc

chmod 600 ~/.netrc  # gives read/write access only to the user, important for login to work!


In [ ]:
%%bash
cd $home # lists home directory
pwd
tail .netrc  # prints content of .netrc file is correct

### Find the correct link on a NASA DAAC server, e.g., GESDISC

1. Go to GESDISC server: https://daac.gsfc.nasa.gov/
2. Search for the GLDAS dataset we are using today: GLDAS_CLSM10_M.2.1
3. Click on the single search result.
4. Go to Web Services (to the right) > OPENDAP
5. Browse to the file you are interested in, e.g., for January 2001
6. Click on nc file for creating an access link (or click on xml file to get some info on the file content)
8. Set Download encoding to NetCDF-4. Leave the remaining form empty.
9. Click on "Copy Data URL", you should receive the following link: https://hydro1.gesdisc.eosdis.nasa.gov/opendap/hyrax/GLDAS/GLDAS_CLSM10_M.2.1/2025/GLDAS_CLSM10_M.A202501.021.nc4
12. Drop the hyrax part of the path, and copy the remaining two parts into the dap_url variable, as shown below.
13. Run the cell to access the dataset with PyDAP.

### Access, inspect the dataset via PyDAP

In [ ]:
from pydap.client import open_url
from pydap.net import create_session


In [ ]:
# 1) Make sure ~/.netrc has Earthdata creds (chmod 600):
# machine urs.earthdata.nasa.gov
#   login YOUR_USERNAME
#   password YOUR_PASSWORD

session = create_session()  # reads ~/.netrc automatically

# DAP4 (recommended on Hyrax)
dap4_url = ("<link chain before hyrax/>"
            "<link chain after hyrax/>")
# dap4_url = ("<link chain without hyrax/")  # alternatively

ds = open_url(dap4_url, session=session, protocol="dap4")
ds

Note that these commands do not download the data yet, but only provide streaming access to them.

Let's first inspect the data.

In [ ]:

# Attributes of the dataset:
ds.attributes # lists the attributes of the dataset
ds.keys()  # lists the variables of the dataset (similar to netCDF4.variables)
#ds.tree()  # shows that this dataset has a shallow hierarchy

# Accessing data variables with formal or lazy syntax
ds['Rainf_tavg'].attributes  # formal syntax
ds.Rainf_tavg.attributes     # lazy syntax (alternatively)

# Info about data variables (here using lazy syntax):
ds.Rainf_tavg.attributes  # attributes of the variable
ds.Rainf_tavg.shape       # shape of the variable
ds.Rainf_tavg.dtype       # data type of the variable


These are all the data types and attributes/methods we cover for now. If you like to know more, check the pydap documentation tutorials at https://pydap.github.io/pydap/en/intro.html.

### Download Data
Finally, let's actually download the data to our computer. Depending on the internet connection, this may take a bit on your computer, but should be pretty fast on Colab.

In [ ]:
# Now we can download the rainfall variable (may take a bit)
#  and its dimensions time, lon, lat, missing_value
rainf_bt = ds.Rainf_tavg[:]     # same as: rainf_bt = ds.['Rainf_tavg'][:]
rainf = ds.Rainf_tavg[:].data
lon = ds.lon[:].data            # same as: lon = ds.['lon'][:]
lat =  ds.lat[:].data
time = ds.time[:].data


Note that the pydap.model.BaseType (above saved in `rainf_bt`) is a wrapper of numpy arrays, which can be accessed with the data attribute (above saved in `rainf`). After executing the code above, the array has been downloaded into “local” memory (RAM) as (uncompressed) numpy arrays.

In [ ]:
# Getting some important additional info

fillVal = rainf_bt.missing_value
unit = rainf_bt.units

fillVal, unit

### Plot the data with Cartopy

We could plot the data like last week with matplotlib, but Cartopy is more sophisticated and allows us to add projections.

In [ ]:
# Prep the data for plotting

# sets fill values to NaN
import numpy as np
rainf[rainf==fillVal] = np.nan  # numpy is required

# convert units to rainfall per month, since we are working with a monthly dataset
rainf = rainf*60*60*24*30
# Note: kg/m2 = mm water equivalent
# Meaning: The area density of 1 kg/m2 of water is equal to a water height of 1 mm over 1 m2 area


Now let's add a plot using cartopy. Our dataset's geographic coordinates are given in the WGS84 (EPSG:4326) reference system, and in Cartopy the correct “data CRS” is Plate Carree.

In [ ]:

import numpy as np
import matplotlib.pyplot as plt
import cartopy.crs as ccrs

fig, ax = plt.subplots(figsize=(6, 4), subplot_kw={"projection": ccrs.PlateCarree()})
im = ax.pcolormesh(lon, lat, rainf[0,:,:], transform=ccrs.PlateCarree())
ax.coastlines(resolution="110m")
plt.colorbar(im, orientation="horizontal")
plt.show()


---
## Exercise A:

1. Retrieve the link and access the file for June 2025 (the latest in the dataset). Inspect the dataset: What are the object types and dimensions of the following variables? Discuss how this is similar and different from handling the netcdf file on your computer with the netCDF4 package, as we did last week.
    - dataset
    - dimensions/coordinate variables (time, lat, lon)
    - packaged data variables
    - actual data content in the variables.
2. Now, download the coordinate variables and the rainfall variable and store them in NumPy arrays. Read also the units, fill values/missing values, and long names of the  variable.
4. Make a plot of the rainfall dataset for 06/2025. How does the rainfall from 06/2025 compare to the rainfall from 01/2025 (which we plotted earlier)?
5. Now let's illustrate how PyDAP helps to get exactly the data we need, e.g. for our study region, without downloading an entire global dataset. Use the following code example to download only the index slice 180:240 for longitude and 20:70 for latitude dimensions. Then make a plot for the variables. From which area on the globe is the dataset?
```python
var = ds.Rainf_tavg                 # does not download anything yet
subset = var[0,bmin:bmax,lmin:lmax] # triggers only fetching of needed bytes
subsetdata   = subset[:].data       # downloads a subsetted numpy array

```
5. Download the global datasets (data arrays) for the following three more variables. Check what their units are? Do not further alter the variables yet, we will use these during the next exercise.
    - Evap_tavg:        *Evapotranspiration*
    - Tair_f_inst:      *Air temperature*
    - SoilMoist_P_inst: *Soil moisture in entire soil colum*



**Optional during class, extra credit for assignment:**

6. Change the projection of one of the global maps to either Robinson or Sinusoidal (or both) by investigating the following resources :
    - List of available projections: https://scitools.org.uk/cartopy/docs/v0.15/crs/projections.html
    - Simple Maps tutorial: https://cartopy.readthedocs.io/stable/matplotlib/intro.html
7. One coordinate dimension of the dataset has only a size of 150. Which is it, and why is it shorter than expected?
8. AI can be extremely helpful for quick debugging of your code (e.g., if you get an error message) or adding some feature without endlessly looking through documentation pages. Let's try two examples:
    - (8a) Use AI (e.g., on Colab or ChatGPT) to find out how to label the colorbar with the unit of the dataset, add gridlines with labels at the x and y axes, and a title. Then apply those and update your last plot.
    - (8b) Use AI to find out how to make two subplots in one plot. Plot the two rainfall datasets for 06/2025 and 01/2025 on top of each other in one figure.


---
## Troubleshooting to continue with the data without PyDAP

### Troubleshooting A.
If access does not work, use the module requests to download the entire .nc file from either the NASA data server, or the github repository (either one of the code cells below).

In [ ]:
## UNCOMMENT TO DOWNLOAD THE FILE from NASA's data server (instead of OPENDAP)

# import requests
# from requests.utils import get_netrc_auth

# url = ("https://data.gesdisc.earthdata.nasa.gov/data/"
#        "GLDAS/GLDAS_CLSM10_M.2.1/2001/GLDAS_CLSM10_M.A200101.021.nc4")
# filename = "GLDAS_CLSM10_M.A200101.021.nc4"

# # Pick up Earthdata credentials from ~/.netrc
# session = requests.Session()
# session.auth = get_netrc_auth(url)

# with session.get(url, stream=True) as r:
#     r.raise_for_status()
#     with open(filename, "wb") as f:
#         for chunk in r.iter_content(chunk_size=1024*1024):
#             if chunk:  # filter out keep-alive chunks
#                 f.write(chunk)

# print("Saved", filename)

In [ ]:
## UNCOMMENT TO DOWNLOAD THE FILE from github

# import requests
# raw_url = "https://raw.githubusercontent.com/GeoPythonVT/geosf25_material/main/data_downloadArchive/GLDAS_CLSM10_M.A202505.021.nc4"
# out = "GLDAS_CLSM10_M.A200101.021.nc4"

# with requests.get(raw_url, stream=True) as r:
#     r.raise_for_status()
#     with open(out, "wb") as f:
#         for chunk in r.iter_content(chunk_size=1024*1024): # use this download version for large files
#             if chunk:
#                 f.write(chunk)

# print("Saved", out)

Continue with data import using the netCDF4 package, as discussed in the previous class:


In [ ]:
# from netCDF4 import Dataset   # on google colab you have to install this
# gldasDat = Dataset('GLDAS_CLSM10_M.A200101.021.nc4')

# [ e for e in gldasDat.dimensions ] # lists all dimensions in the dataset
# [ e for e in gldasDat.variables ] # lists all variables in the dataset (here muted)
# gldasDat.variables['Rainf_tavg']
# gldasDat_rainf = gldasDat.variables['Rainf_tavg'][:].data
# gldasDat_rainf

Alternatively (e.g., on Colab) you can also use xarray module to import the netcdf dataset as follows:

In [ ]:
# import xarray as xr

# # Point to the file you downloaded
# gldasDat = xr.open_dataset("GLDAS_CLSM10_M.A200101.021.nc4")

# print(ds)          # summary of variables, dimensions, attributes
# print(ds.data_vars)

# gldasDat_rainf = ds["Rainf_tavg"].values
# gldasDat_rainf


### Troubleshooting B.
If you still have problem, but want to follow the next steps in class without interruption, download the file manually as follows:

1. Go to the data request form page:
    - https://hydro1.gesdisc.eosdis.nasa.gov/opendap/hyrax/GLDAS/GLDAS_CLSM10_M.2.1/2025/GLDAS_CLSM10_M.A202501.021.nc4.dmr.html
    - https://hydro1.gesdisc.eosdis.nasa.gov/opendap/hyrax/GLDAS/GLDAS_CLSM10_M.2.1/2025/GLDAS_CLSM10_M.A202506.021.nc4.dmr.html
2. Set Download encoding to NetCDF-4. Leave the remaining form empty (or better choose only the physical variables we are interested in).
3. Click on "Get Data"
4. Save the nc file in an accessible folder.
5. Continue with data import using the netCDF4 package, as discussed in the previous class.


### Additional Note
You can also reduce the number of variables or size of arrays that you access from the beginning, if you have copied the respective access link from the OpenDAP Data Request form. For example, the following alteration retrieves only acces to a sliced Rainf variable and the lat and lon arrays:

In [ ]:

# session = create_session()  # reads ~/.netrc automatically
# dap4_url = ("https://hydro1.gesdisc.eosdis.nasa.gov/opendap/"
#             "GLDAS/GLDAS_CLSM10_M.2.1/2025/GLDAS_CLSM10_M.A202501.021.nc4?dap4.ce=/lat[0:1:149];/lon[0:1:359];/Rainf_tavg[0:1:0][0:1:149][0:1:359]")
# ds = open_url(dap4_url, session=session, protocol="dap4")
# rainf = ds.Rainf_tavg[:].data # download data...
# print(rainf.shape)


---
## Part B: Descriptive (Univariate) Statistics & Outlier Detection

To continue this part, make sure you have downloaded the following variables from Part A:

- Evap_tavg        *Evapotranspiration*
- Tair_f_inst      *Air temperature*
- SoilMoist_P_inst *Soil moisture in entire soil colum*


<div class="alert alert-warning">
    
**NOTE**
In case you started a new session, you have to again install the package pydap and download the data, as we did above. 
</div>

In this tutorial, we will look into spatial statistics of the variables. Note, that the tools work the same and can also be used to analyze behaviour in the temporal domain (e.g., for time series of air temperature at a given location).


Now, let's begin. 

What should we expect from the data?
   + It may have many outliers
   + What are outliers?
          + e.g., soil moisture end evaporation cannot be negative
   + We need to be aware of the feasible range of the numerical values

Handling Outliers and Missing Values are two common, but non-trivial problems in data science:ß
   + Simplest way: visual data exploration
   + set threashold to remove outliers


### Histogram

Let's plot an histogram for evaporation.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns


In [ ]:
# Basic histogram
fig = plt.figure(figsize=(4, 3))
sns.histplot(evap.ravel(), bins=30, kde=False)   # kde=False disables the smooth density curve
# > use array.ravel() to convert tensor/matrix into a data vector and create one histogram for entire map
# works also with plt.hist(data), but seaborn has better styling features
plt.title("Histogram of Evaporation")
plt.xlabel("Evaporation (mm per sec)")
plt.ylabel("Count")
plt.show()

The negative values are all outliers (here actually the dataset's fill values over the oceans). Let's set them to NaN.
  + Most packages (numpy, pandas, seaborn) will ignore NaNs when doing statistical analysis (mean, std, covariance, correlation, etc.)
  + Let's use basic physical understanding of the variables and simple thresholds:
       + Rainfall has to be positive
       + Air temerapture: smaller 0 Kelvin degrees are regarded as outliers
       + Soil moisture: 0~4000 kg/m2 are regarded as inliers
       + Negative Evaporation values exist (and refer to condensation), but should be small, a threshold of -100 kg/m2/s for outliers is safe

Let's remove the outliers for all, but then we will continue to work only with soil moisture.

In [ ]:
# removing the outliers for rainf, tair, soilm and evap

rainf[rainf<0]=np.nan  # example for rainfall
tair[tair<0]=np.nan    # example for rainfall

# add code for removing outliers in soilm
# add code for removing outliers evaporation

#### Questions:
- How does the histogram for soil moisture look like after removing the outliers? 
- Experiment with the number of bins. What happens if the parameter is too low/high?
- What do you consider an optimal value?

In [ ]:
fig = plt.figure(figsize=(4, 3))
sns.histplot(soilm.ravel(), bins=8, kde=False)
plt.xlabel("Soil Moisture")
plt.ylabel("Count")
plt.show()

### Mean, Median, Standard Deviation, Percentiles

Basic NumPy functions are useful to calculate these:

In [ ]:

# these functions ignore all nan
soilMin  = np.nanmin(soilm)
soilMax  = np.nanmax(soilm)
soilMean = np.nanmean(soilm)
soilMedian = np.nanmedian(soilm)
soilStd  = np.nanstd(soilm)
soilQ1, soilQ3 = np.nanpercentile(soilm,[25, 75]) # you can use this to estimate any other percentile
soilIQR = soilQ3 - soilQ1

print("Min: ", round(soilMin,1), "mm w.eq.") # unit kg/m2 is equal to mm of a water column
print("Max: ", round(soilMax,1), "mm w.eq.") 
print("Mean / Median / Std:", round(soilMean,1), round(soilMedian,1), round(soilStd,1))
print("Q1 / Q2:", round(soilQ1,1), round(soilQ3,1))
print("IQR:", round(soilIQR,1))


#### Task
Discuss how you could calculate the 90th percentile of the variable?

In [ ]:
# What is the 90th percentile of soil moisture?


### Boxplot

Let's make a boxplot for soil moisture values. For that, it is helpful to store it in Pandas data frames, to handle the data easier in seaborn. To learn more about pandas dataframes, read the optional tutorial B08 on Github.
Note that the nan values are automatically ignored.

In [ ]:
# Store data in Pandas Dataframe
import numpy as np, pandas as pd

A = np.array([soilm.ravel()/10]) # converting units to cm 
AcolNames = ["Soil Moisture"]   # name the column
df = pd.DataFrame(A.T, columns=AcolNames)  


In [ ]:
# Make Boxplot
fig = plt.figure(figsize=(2, 3))
sns.boxplot(data = df)
plt.ylabel("soil moisture (cm)")
plt.show()


#### Task
Briefly discuss: What elements does the boxplot show? 

We can also make a boxplot of two variables at the same time:

In [ ]:
B = np.array([soilm.ravel()/10,tair.ravel()-272.15]) # converting units to cm / Celcius
BcolNames = ["Soil Moisture", "Air Temperature"]   # name the columns
df2 = pd.DataFrame(B.T, columns=BcolNames)  

In [ ]:
# Make Boxplot
fig = plt.figure(figsize=(3, 3))
sns.boxplot(data = df2)
plt.ylabel("soilm (cm) / tair (C)")
plt.show()

#### Task
Briefly discuss: Why does the air temperature variable indicate any values beyond the wiskers?


In [ ]:
# Where are the whisker ends? (this is the standard setting in seaborn, it can be changed)
soilWtop    = min(soilMax, soilQ3 + 1.5*soilIQR)
soilWbottom = max(soilMin, soilQ1 - soilIQR)

print("soilWbottom / soilWtop:", round(soilWbottom,1), round(soilWtop,1))

### Smooth probability densitity function and distribution

Next, we will use seaborn to add a empirical density function (or kernel density estimate KDE) to our histogram. Then, let's assume our dataset follows a normal distribution.
We can use the scipy package to fit our dataset to this distribution as follows:


In [ ]:
import scipy.stats as stats

data = soilm[~np.isnan(soilm)]

# Fit a normal distribution
mu, sigma = stats.norm.fit(data)

print(f"Estimated mean = {mu:.2f}, std = {sigma:.2f}")

# Plot histogram + fitted PDF
fig = plt.figure(figsize=(4, 3))

# plots the histogram and empirical density function:
sns.histplot(data, bins=30, kde=True, stat='density', label='Histogram') 

# plots fitted normal distribution:
x = np.linspace(min(data), max(data), 100)
plt.plot(x, stats.norm.pdf(x, mu, sigma), 'r-', lw=2, label='Fitted Normal') 

plt.legend()
plt.show()

#### Task
How do the histogram, the KDE and the fitted normal distribution deviate from each other? Discuss if the dataset is actually normal distributed? 

--- 
## Exercise B

1. Estimate mean, median, standard deviation, upper and lower quartiles for the rainfall variable in units of cm per month.
2. Make boxplots for rainfall and evaporation in one graph, after converting both to cm per month. Label correctly.
3. Plot a histogram and density plot for the rainfall variable in units of cm per month.

Extra Credit

4. Using seaborn, find out how to show the mean in the boxplot, in addition to the median.
5. Find out how to fit a skewed distribution (e.g., Skew-Normal, or another one) to the soilm dataset that fits better than the normal distribution and plot an example.

I encourage you to use AI to help with the latter two tasks, either directly on Colab, or via ChatGPT, or similar.


---

---
## Part C: Multivariate Statistics

We will continue to work with the rainfall, evaporation, air temperature and soil moisture dataset and assess multi-variate statistics in the spatial domain.

### Pearson and Spearman correlation

These standard bi-variate values can be estimated with various Python packages, including numpy, scipy, pandas. 

For the bivariate case, scipy is most straightforward:

In [ ]:
from scipy import stats
m = np.isfinite(rainf) & np.isfinite(evap)    # filter for nan values in either array

r,  p = stats.pearsonr(rainf[m], evap[m])     # linear, parametric correlation (does not have nan-policy, needs filter)
rho, p = stats.spearmanr(rainf.ravel(), evap.ravel(), nan_policy='omit')   # rank (monotonic, nonparametric)

print(round(r,2), round(rho,2))

### Scatter plot and Regression Lines

A scatter plot provides a more detailed insight into the correlation between two variables. This can be done very promptly using matplotlib.

In [ ]:

x = rainf[m]*60*60*24*30/10
y = evap[m]*60*60*24*30/10

fig = plt.figure(figsize=(4, 3))
plt.scatter(x,y, s=20, alpha=0.7)   # s=size, alpha=transparency
plt.xlabel("Rainfall (cm/month)"); plt.ylabel("Evaporation (cm/month)")
plt.title("Scatter")
plt.show()


You can also use seaborn. A bit more sophisticated to code, but a nicer plot.

In [ ]:
# Convert data in a pandas dataframe
A = np.array([x,y])  
AcolNames = ["x","y"]  
df = pd.DataFrame(A.T, columns=AcolNames)  

# Seaborn scatterplot
fig = plt.figure(figsize=(4, 3))
sns.scatterplot(data=df, x="x", y="y")                         # basic
plt.xlabel("Rainfall (cm/month)"); plt.ylabel("Evaporation (cm/month)")
plt.title("Scatter")
plt.show()


Seaborn also has a built in option to add a trendline for the scatter. We will continue talking about regression in another lecture.

In [ ]:
# Regression

fig = plt.figure(figsize=(4, 3))
sns.scatterplot(data=df, x="x", y="y")                         # basic
sns.regplot(data=df, x="x", y="y", scatter_kws=dict(alpha=0.6), line_kws=dict(color="yellow"))
plt.xlabel("Rainfall (cm/month)"); plt.ylabel("Evaporation (cm/month)")
plt.title("Scatter")
plt.show()

### Heatmaps of correlation 

To display multiple bivariate correlations in one graph, heatmaps are very useful.

In [ ]:

# Store all data in Pandas Dataframe
A = np.array([ rainf.ravel()*60*60*24*30/10,
               evap.ravel()*60*60*24*30/10,
               soilm.ravel()/10])
AcolNames = ["Rainf", "Evap","SoilM"]   # name the columns
df = pd.DataFrame(A.T, columns=AcolNames)  

# pairwise Pearson by default
corr = df.corr(numeric_only=True)           
#mask = np.triu(np.ones_like(corr, dtype=bool))  # hide upper triangle (optional)

# make plot
plt.figure(figsize=(4,3))
sns.heatmap(corr, annot=True, fmt=".2f", #mask=mask, 
            vmin=-1, vmax=1, square=True, cmap="vlag", cbar_kws={"label":"Correlation"})
plt.title("Correlation heatmap")
plt.show()

### Pairplots

Pair plots (aka a scatterplot matrix or SPLOM) are grids of plots that show every pairwise relationship between numerical variables (scatterplots off-diagonal) and each variable’s univariate distribution on the diagonal (hist/KDE), useful for spotting correlation, clusters, outliers, and nonlinearity.

In [ ]:
plt.figure(figsize=(4,4))
sns.pairplot(df, vars=["Rainf", "Evap","SoilM"], corner=False)
# sns.pairplot(df, vars=["Rainf", "Evap","SoilM"], diag_kind="kde", corner=False) # KDE instead of histogram
plt.show()

### Supplement: Violinplot

In [ ]:
# Use seaborn to make a violinplot
fig = plt.figure(figsize=(4, 2))
sns.violinplot(df)
plt.show()

#### Task
Find out what a violinplot represents.

--- 
## Exercise C

1. Generate a heatmap and pairplots for all four variables.
2. Which variable is the least correlated to all other variables? Discuss whether you expected that and what do you think might be an explanation for this finding? Also, make a plot showing mean value per latitude for all four variables. Alter the units, so you can see variablility four all four graphs at the same time. How do you interpret the result? Include this in the discussion. 
3. What additional information might the pairplots provide, that the heatmaps cannot?

Extra credit

4. Mask the upper triangle of the heatmap plot (by uncommenting that parameter in the code example). Which version do you prefer, and why?
5. Generate a heatmap for spearman correlation, instead of pearson. Discuss why the results are different.
